<a href="https://colab.research.google.com/github/jcmachicao/knowledge_engineering/blob/main/cur_KE___RAG_Demo_Embeddings_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demostración de Retrieval-Augmented Generation (RAG)

Construcción de embeddings, índice FAISS y consulta RAG sobre `patient_documents.json`.

In [1]:
!pip install -q sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.6 MB/s eta 0:00:00


FAISS: Facebook AI Similarity Search

In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [4]:
from google.colab import userdata
client = OpenAI(api_key=userdata.get('OAIK_JCMV'))

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
ruta = "drive/My Drive/00 2026_all/2026 Cursos/RAGvsCogWorkflow/"
os.listdir(ruta)

['dataset_protocolos_clinicos.json',
 'dataset_protocolos_clinicos_estandar.json',
 'merged_protocolos_clinicos.json',
 'patient_documents.json',
 'cur_KE__generador_textos.ipynb',
 'cur_KE___RAG_Demo_Embeddings_FAISS.ipynb']

In [8]:
with open(ruta + "patient_documents.json","r",encoding="utf8") as f:
    docs = json.load(f)

texts = [d["story"] for d in docs]
ids = [d["id"] for d in docs]
print(f"Documentos: {len(texts)}")

Documentos: 25


In [9]:
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")

print(embeddings.shape)

(25, 384)


In [10]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Vectores indexados:", index.ntotal)

Vectores indexados: 25


In [11]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7e63a4115b70> >

In [12]:
query = "¿Qué pacientes podrían requerir el protocolo OMEGA?"

In [13]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

In [15]:
k = 5
distances, indices = index.search(query_embedding, k)
context = ""

for i, idx in enumerate(indices[0]):
    print(ids[idx])
    print(texts[idx])
    context += f"\nDocumento {i+1}\n{texts[idx]}\n"

P011
Soy un paciente de 71 años, de sexo femenino. He presentado fiebre alta desde hace aproximadamente dos días. También me cuesta respirar y me agito con facilidad. No tengo antecedentes de diabetes. Tengo antecedentes de hipertensión. Los análisis recientes indicaron una función renal conservada. Nunca he presentado reacciones alérgicas relevantes. Actualmente utilizo: Losartan. Inicio brusco de fiebre alta y dificultad respiratoria. Por este motivo solicité atención médica.
P015
Soy un paciente de 69 años, de sexo femenino. La fiebre comenzó hace 48 horas y no ha cedido completamente. También me cuesta respirar y me agito con facilidad. No tengo antecedentes de diabetes. Tengo antecedentes de hipertensión. Según el médico, la función renal es adecuada. No conozco alergias medicamentosas. Actualmente utilizo: Amlodipine. Compromiso respiratorio moderado. El personal médico decidió realizar una evaluación completa.
P021
Paciente de 58 años (masculino). He presentado fiebre alta desde

In [16]:
prompt = f'''
Eres un asistente médico.
Utiliza únicamente la información contenida en los documentos recuperados.

DOCUMENTOS
{context}

PREGUNTA
{query}

Si la información es insuficiente, indícalo explícitamente.
'''

In [17]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)

La información proporcionada en los documentos recuperados no define explícitamente qué criterios o condiciones ameritan el protocolo OMEGA.

Sin embargo, analizando los casos presentados:

- Pacientes con fiebre alta y dificultad respiratoria (Documento 1, 2, 3, 5)
- Edad avanzada (Documento 1, 2)
- Presencia de comorbilidades como hipertensión (Documento 1, 2) o diabetes (Documento 5)
- Presencia de fiebre y síntomas respiratorios moderados a severos.

Estos factores suelen ser indicativos para protocolos médicos que atienden pacientes con infección respiratoria y riesgo de complicaciones.

Dado que no se especifica en los documentos qué pacientes califican para el protocolo OMEGA, no es posible determinar con certeza cuáles pacientes requieren dicho protocolo.

**Conclusión:**  
La información proporcionada es insuficiente para identificar qué pacientes podrían requerir el protocolo OMEGA.


In [18]:
for k in [2,4,6,10]:
    print("="*60)
    print("TOP K =", k)
    distances, indices = index.search(query_embedding, k)
    print([ids[idx] for idx in indices[0]])

TOP K = 2
['P011', 'P015']
TOP K = 4
['P011', 'P015', 'P021', 'P023']
TOP K = 6
['P011', 'P015', 'P021', 'P023', 'P016', 'P022']
TOP K = 10
['P011', 'P015', 'P021', 'P023', 'P016', 'P022', 'P008', 'P009', 'P020', 'P019']


## Conclusión

El LLM responde únicamente utilizando los documentos recuperados por el índice vectorial; no utiliza toda la base documental.